# Linkography Code Annotation Implementation

## TRY: Detect ONE linkography pattern using ONE or TWO annotation codes

Remember, these sessions' JSON files were split into 10 minute segments. Might have to fix this again if we are to scale implementation etc.

## Iteration 1

In [7]:
import json
from pathlib import Path

# --------------------------------------------------------
# 1. Load the JSON file
# --------------------------------------------------------
file_path = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data/2021ABI/session_data/2021_05_21_ABI_S4.json")

with open(file_path, "r") as f:
    data = json.load(f)

print("Top-level keys:", data.keys())

# --------------------------------------------------------
# 2. Normalize to utterance list
# --------------------------------------------------------
# In this file, utterances live inside "all_data"
raw_utts = data["all_data"]

print("Number of utterances:", len(raw_utts))
print("Example utterance keys:", raw_utts[0].keys())

# --------------------------------------------------------
# 3. Extract the fields we need
#    (use index as a fallback utterance_id)
# --------------------------------------------------------
utterances = []
for i, u in enumerate(raw_utts):
    utterances.append({
        "utterance_id": u.get("utterance_id", f"utt_{i}"),
        "speaker": u.get("speaker"),
        "timestamp": u.get("timestamp"),   # could be None here; that's fine for now
        "annotations": u.get("annotations", {}) or {},
        "transcript": u.get("transcript", "")
    })

# Quick sanity check: how many have non-empty annotations?
non_empty = sum(1 for u in utterances if u["annotations"])
print("Utterances with non-empty annotations:", non_empty)

# Show a couple of examples so you can eyeball them
for u in utterances[:3]:
    print("\n--- Example utterance ---")
    print("utterance_id:", u["utterance_id"])
    print("speaker:", u["speaker"])
    print("timestamp:", u["timestamp"])
    print("annotations keys:", list(u["annotations"].keys()))
    print("snippet:", u["transcript"][:120])


# --------------------------------------------------------
# 4. Define heuristic for “short-span forelink”
#    We treat certain annotation categories as "builds on / responds to"
#    and link each such utterance to the *immediately previous* one.
# --------------------------------------------------------
def detect_short_span_forelinks(utterances):
    links = []

    # Categories that usually indicate a move that builds on
    # or responds to prior talk (you can tweak this set)
    build_categories = [
        "Knowledge Sharing",
        "Coordination and Decision Practices",
        "Participation Dynamics",
        "Communication Practices",
        "Goal/Focus Clarity",
        "Trust and Psychological Safety",
        "Conflict Management"
    ]

    for i in range(1, len(utterances)):
        prev_u = utterances[i - 1]
        curr_u = utterances[i]

        ann_curr = curr_u.get("annotations", {}) or {}

        # If the current utterance has *any* of these categories,
        # treat it as a short-span forelink from previous → current
        if any(cat in ann_curr for cat in build_categories):
            links.append({
                "from": prev_u["utterance_id"],
                "to": curr_u["utterance_id"],
                "type": "short_span_forelink",
                "speaker_from": prev_u["speaker"],
                "speaker_to": curr_u["speaker"],
                "categories_curr": list(ann_curr.keys()),
                "snippet_prev": (prev_u.get("transcript") or "")[:120],
                "snippet_curr": (curr_u.get("transcript") or "")[:120],
            })

    return links


# --------------------------------------------------------
# 5. Run detection and inspect a few links
# --------------------------------------------------------
links_found = detect_short_span_forelinks(utterances)

print(f"\nDetected {len(links_found)} short-span forelinks.")
for link in links_found[:10]:
    print("\n--- Forelink ---")
    for k, v in link.items():
        print(f"{k}: {v}")

Top-level keys: dict_keys(['all_speakers', 'total_speaking_length', 'all_data'])
Number of utterances: 259
Example utterance keys: dict_keys(['speaker', 'timestamp', 'transcript', 'speaking_duration', 'nods_others', 'smile_self', 'smile_other', 'distracted_others', 'hand_gesture', 'interuption', 'overlap', 'screenshare', 'screenshare_content', 'start_time', 'end_time', 'annotations', 'role', 'when'])
Utterances with non-empty annotations: 259

--- Example utterance ---
utterance_id: utt_0
speaker: Anna Moore
timestamp: 00:00-00:23
annotations keys: ['Coordination and Decision Practices', 'Information Seeking']
snippet: suggest is that we very quickly introduce ourselves um and then we'll start working on generating ideas. So is there any

--- Example utterance ---
utterance_id: utt_1
speaker: Anna Moore
timestamp: 00:23-01:57
annotations keys: ['Knowledge Sharing', 'Coordination and Decision Practices']
snippet: So I thought I thought we all first facilitators introduce themselves. But

## Iteration 2

In [8]:
import json
from pathlib import Path

# --------------------------------------------------------
# 1. Load the JSON file.  2021-05-21-ABI-S4
# --------------------------------------------------------
file_path = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data/2021ABI/session_data/2021_05_21_ABI_S4.json")

with open(file_path, "r") as f:
    data = json.load(f)

print("Top-level keys:", data.keys())

# --------------------------------------------------------
# 2. Normalize to utterance list
# --------------------------------------------------------
raw_utts = data["all_data"]

print("Number of utterances:", len(raw_utts))
print("Example utterance keys:", raw_utts[0].keys())

# --------------------------------------------------------
# 3. Extract the fields we need
# --------------------------------------------------------
utterances = []
for i, u in enumerate(raw_utts):
    utterances.append({
        "utterance_id": u.get("utterance_id", f"utt_{i}"),
        "speaker": u.get("speaker"),
        "timestamp": u.get("timestamp"),
        "annotations": u.get("annotations", {}) or {},
        "transcript": u.get("transcript", "")
    })

non_empty = sum(1 for u in utterances if u["annotations"])
print("Utterances with non-empty annotations:", non_empty)

# --------------------------------------------------------
# 4. Helper: does this utterance look "idea-building"?
# --------------------------------------------------------
IDEA_CODES = {
    "Idea Management",
    "Knowledge Sharing",
    "Evaluation Practices",
    "Integration Practices",
}

def is_idea_building(ann_dict):
    """
    ann_dict: {"CodeName": {"explanation": ..., "score": ..., "when": ...}, ...}
    Returns True if the utterance has at least one idea-related code
    that is NOT just an intro / "beginning" meta move.
    """
    for code, info in ann_dict.items():
        if code in IDEA_CODES:
            when = (info or {}).get("when", "").lower()
            if when != "beginning":   # exclude introductions
                return True
    return False

# --------------------------------------------------------
# 5. Detect short-span forelinks *only for idea-building moves*
# --------------------------------------------------------
def detect_short_span_forelinks_v2(utterances):
    links = []

    for i in range(1, len(utterances)):
        prev_u = utterances[i - 1]
        curr_u = utterances[i]

        ann_curr = curr_u.get("annotations", {}) or {}

        # New heuristic: only link if current utterance is "idea-building"
        if is_idea_building(ann_curr):
            links.append({
                "from": prev_u["utterance_id"],
                "to": curr_u["utterance_id"],
                "type": "short_span_forelink_v2",
                "speaker_from": prev_u["speaker"],
                "speaker_to": curr_u["speaker"],
                "timestamp_curr": curr_u["timestamp"],
                "codes_curr": list(ann_curr.keys()),
                "snippet_prev": (prev_u.get("transcript") or "")[:120],
                "snippet_curr": (curr_u.get("transcript") or "")[:120],
            })

    return links

# --------------------------------------------------------
# 6. Run detection and compare
# --------------------------------------------------------
links_v2 = detect_short_span_forelinks_v2(utterances)

print(f"\nIteration 2: detected {len(links_v2)} idea-building short-span forelinks.")
for link in links_v2[:10]:
    print("\n--- Forelink v2 ---")
    for k, v in link.items():
        print(f"{k}: {v}")

Top-level keys: dict_keys(['all_speakers', 'total_speaking_length', 'all_data'])
Number of utterances: 259
Example utterance keys: dict_keys(['speaker', 'timestamp', 'transcript', 'speaking_duration', 'nods_others', 'smile_self', 'smile_other', 'distracted_others', 'hand_gesture', 'interuption', 'overlap', 'screenshare', 'screenshare_content', 'start_time', 'end_time', 'annotations', 'role', 'when'])
Utterances with non-empty annotations: 259

Iteration 2: detected 127 idea-building short-span forelinks.

--- Forelink v2 ---
from: utt_67
to: utt_68
type: short_span_forelink_v2
speaker_from: Benjamin Bartelle
speaker_to: Benjamin Bartelle
timestamp_curr: 06:09-07:08
codes_curr: ['Knowledge Sharing', 'Information Seeking', 'Relational Climate']
snippet_prev: I think some of us are you can study people though, right?
snippet_curr: Um Molly, you study people, but what you get is not mechanistic per se. It's like a convolution of the mechanisms. And s

--- Forelink v2 ---
from: utt_68
to: u

# Stopping the Iteration Implementation. Now Reflecting on it all